In [1]:
from langchain_community.document_loaders import TextLoader
from chunking import chunk_text


# --------------------------------------------------
# LOAD KNOWLEDGE BASE
# --------------------------------------------------

filename = "C:\\Users\\priya\\genai_classes_project\\DSA_coach_proj\\knowledge_base\\fivehundredwords_dsa.txt"

loader = TextLoader(filename, encoding="utf-8")
docs = loader.load()

text_content = docs[0].page_content


# --------------------------------------------------
# GENERATE CHUNKS USING ALL 3 METHODS
# --------------------------------------------------

chunks_dict = chunk_text(text_content)


# --------------------------------------------------
# EVALUATE CHUNKING METHODS
# --------------------------------------------------

print("\n" + "=" * 60)
print("           CHUNKING EVALUATION")
print("=" * 60)


for method, chunks in chunks_dict.items():

    print("\n" + "-" * 60)

    if method == "recursive":
        method_name = "RecursiveCharacterTextSplitter"

    elif method == "character":
        method_name = "CharacterTextSplitter"

    else:
        method_name = "TokenTextSplitter"

    print(f"Method: {method_name}")
    print("-" * 60)

    # Number of chunks
    number_of_chunks = len(chunks)

    # Character lengths
    chunk_lengths = [len(chunk) for chunk in chunks]

    average_length = sum(chunk_lengths) / len(chunk_lengths)

    minimum_length = min(chunk_lengths)
    maximum_length = max(chunk_lengths)

    total_characters = sum(chunk_lengths)

    print(f"Number of chunks:       {number_of_chunks}")
    print(f"Average chunk length:   {average_length:.2f} characters")
    print(f"Minimum chunk length:   {minimum_length} characters")
    print(f"Maximum chunk length:   {maximum_length} characters")
    print(f"Total characters:       {total_characters}")


print("\n" + "=" * 60)
print("Evaluation completed.")
print("=" * 60)

C:\Users\priya\AppData\Local\Temp\ipykernel_14516\2141914932.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\priya\genai_classes_project\DSA_coach_proj\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔹 RecursiveCharacterTextSplitter produced: 16 chunks
🔹 CharacterTextSplitter produced: 16 chunks
🔹 TokenTextSplitter produced: 6 chunks

           CHUNKING EVALUATION

------------------------------------------------------------
Method: RecursiveCharacterTextSplitter
------------------------------------------------------------
Number of chunks:       16
Average chunk length:   393.19 characters
Minimum chunk length:   316 characters
Maximum chunk length:   498 characters
Total characters:       6291

------------------------------------------------------------
Method: CharacterTextSplitter
------------------------------------------------------------
Number of chunks:       16
Average chunk length:   391.00 characters
Minimum chunk length:   315 characters
Maximum chunk length:   494 characters
Total characters:       6256

------------------------------------------------------------
Method: TokenTextSplitter
------------------------------------------------------------
Number of chunks

In [8]:
import numpy as np
import pandas as pd
from langchain_community.document_loaders import TextLoader
from model import embeddings
from chunking import chunk_text

In [9]:
filename ="C:\\Users\\priya\\genai_classes_project\\DSA_coach_proj\\knowledge_base\\fivehundredwords_dsa.txt"

loader = TextLoader(filename, encoding="utf-8")
docs = loader.load()

text_content = docs[0].page_content

# Generate all three chunking strategies
chunks_dict = chunk_text(text_content)

print("Knowledge base loaded successfully.")
print(f"Total characters: {len(text_content)}")

🔹 RecursiveCharacterTextSplitter produced: 16 chunks
🔹 CharacterTextSplitter produced: 16 chunks
🔹 TokenTextSplitter produced: 6 chunks
Knowledge base loaded successfully.
Total characters: 6311


In [4]:
for method, chunks in chunks_dict.items():
    print("\n" + "=" * 70)
    print(f"{method.upper()} CHUNKS")
    print("=" * 70)

    for i, chunk in enumerate(chunks):
        print(f"\n--- Chunk {i + 1} ---")
        print(chunk[:500])


RECURSIVE CHUNKS

--- Chunk 1 ---
DSA FUNDAMENTALS KNOWLEDGE BASE

1. ARRAYS

An array is a linear data structure used to store multiple elements in a collection. In a traditional array, elements are stored in contiguous memory locations. Each element can be accessed using an index. Most programming languages use zero-based indexing, meaning the first element is at index 0.

--- Chunk 2 ---
For example, an array containing 10, 20, 30 and 40 has 10 at index 0, 20 at index 1, 30 at index 2 and 40 at index 3. Accessing an element using its index takes O(1) time.

Common operations on arrays include traversal, searching, insertion, deletion and updating. Searching an unsorted array generally takes O(n) time. Inserting or deleting an element from the middle may take O(n) time because other elements may need to be shifted.

--- Chunk 3 ---
Arrays are useful when fast random access is required. They are commonly used to implement other data structures such as stacks, heaps and dynamic arrays

In [10]:
evaluation_questions = [
    "What is an array?",
    "What is a linked list?",
    "What is a stack?",
    "What is a queue?",
    "What is binary search?",
    "What is the time complexity of binary search?",
    "What is a tree?",
    "What is a binary tree?",
    "What is a graph?",
    "What is the difference between an array and a linked list?"
]

print(f"Number of evaluation questions: {len(evaluation_questions)}")

Number of evaluation questions: 10


In [11]:
# ============================================================
# CREATE EMBEDDINGS FOR ALL CHUNKING METHODS
# ============================================================

chunk_embeddings = {}

for method, chunks in chunks_dict.items():

    print(f"\nCreating embeddings for: {method}")

    vectors = []

    for i, chunk in enumerate(chunks):

        vector = embeddings.embed_query(chunk)
        vectors.append(np.array(vector))

    chunk_embeddings[method] = np.array(vectors)

    print(
        f"{method}: "
        f"{len(vectors)} embeddings created"
    )


Creating embeddings for: recursive
recursive: 16 embeddings created

Creating embeddings for: character
character: 16 embeddings created

Creating embeddings for: token
token: 6 embeddings created


In [12]:
# ============================================================
# RETRIEVAL FUNCTION
# ============================================================

def retrieve_top_k(query, chunks, vectors, top_k=5):

    query_vector = np.array(
        embeddings.embed_query(query)
    )

    query_norm = np.linalg.norm(query_vector)

    scores = []

    for i, vector in enumerate(vectors):

        vector_norm = np.linalg.norm(vector)

        similarity = np.dot(
            query_vector,
            vector
        ) / (
            query_norm * vector_norm
        )

        scores.append({
            "chunk_id": i,
            "score": float(similarity),
            "content": chunks[i]
        })

    scores.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scores[:top_k]

In [13]:
# ============================================================
# RUN RETRIEVAL FOR ALL QUESTIONS
# ============================================================

retrieval_results = []

for question in evaluation_questions:

    for method in chunks_dict:

        results = retrieve_top_k(
            question,
            chunks_dict[method],
            chunk_embeddings[method],
            top_k=5
        )

        for rank, result in enumerate(results, start=1):

            retrieval_results.append({
                "question": question,
                "method": method,
                "rank": rank,
                "chunk_id": result["chunk_id"],
                "score": result["score"]
            })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df.head(20)

,question,method,rank,chunk_id,score
0,What is an array?,recursive,1,0,0.686999
1,What is an array?,recursive,2,1,0.656812
2,What is an array?,recursive,3,2,0.642147
3,What is an array?,recursive,4,6,0.627923
4,What is an array?,recursive,5,5,0.612902
5,What is an array?,character,1,0,0.687894
6,What is an array?,character,2,1,0.659191
7,What is an array?,character,3,2,0.647399
8,What is an array?,character,4,6,0.627380
9,What is an array?,character,5,5,0.617402


In [14]:
for method, chunks in chunks_dict.items():

    print("\n" + "=" * 80)
    print(f"{method.upper()} — CHUNKS")
    print("=" * 80)

    for i, chunk in enumerate(chunks):
        print(f"\n--- CHUNK {i} ---")
        print(chunk)


RECURSIVE — CHUNKS

--- CHUNK 0 ---
DSA FUNDAMENTALS KNOWLEDGE BASE

1. ARRAYS

An array is a linear data structure used to store multiple elements in a collection. In a traditional array, elements are stored in contiguous memory locations. Each element can be accessed using an index. Most programming languages use zero-based indexing, meaning the first element is at index 0.

--- CHUNK 1 ---
For example, an array containing 10, 20, 30 and 40 has 10 at index 0, 20 at index 1, 30 at index 2 and 40 at index 3. Accessing an element using its index takes O(1) time.

Common operations on arrays include traversal, searching, insertion, deletion and updating. Searching an unsorted array generally takes O(n) time. Inserting or deleting an element from the middle may take O(n) time because other elements may need to be shifted.

--- CHUNK 2 ---
Arrays are useful when fast random access is required. They are commonly used to implement other data structures such as stacks, heaps and dynamic arra

In [15]:
ground_truth = {
    "What is an array?": {
        "recursive": [0],
        "character": [0],
        "token": [0]
    },

    "What is a linked list?": {
        "recursive": [2],
        "character": [2],
        "token": [0, 1]
    },

    "What is a stack?": {
        "recursive": [4],
        "character": [4],
        "token": [1, 2]
    },

    "What is a queue?": {
        "recursive": [5],
        "character": [5],
        "token": [2]
    },

    "What is binary search?": {
        "recursive": [6, 7],
        "character": [6, 7],
        "token": [2, 3]
    },

    "What is the time complexity of binary search?": {
        "recursive": [8],
        "character": [8],
        "token": [3]
    },

    "What is a tree?": {
        "recursive": [13],
        "character": [13],
        "token": [4, 5]
    },

    "What is a binary tree?": {
        "recursive": [13],
        "character": [13],
        "token": [4, 5]
    },

    "What is a graph?": {
        "recursive": [14],
        "character": [14],
        "token": [5]
    },

    "What is the difference between an array and a linked list?": {
        "recursive": [0, 2, 4],
        "character": [0, 2, 4],
        "token": [0, 1, 2]
    }
}

In [16]:
def calculate_metrics(
    evaluation_questions,
    chunks_dict,
    chunk_embeddings,
    ground_truth,
    k_values=(1, 3, 5)
):
    metrics = []

    for method in chunks_dict:

        for k in k_values:

            hits = 0
            reciprocal_ranks = []

            for question in evaluation_questions:

                expected_chunks = set(
                    ground_truth[question][method]
                )

                results = retrieve_top_k(
                    question,
                    chunks_dict[method],
                    chunk_embeddings[method],
                    top_k=k
                )

                retrieved_ids = [
                    r["chunk_id"]
                    for r in results
                ]

                # -------------------------
                # Recall@K
                # -------------------------

                if expected_chunks.intersection(retrieved_ids):
                    hits += 1

                # -------------------------
                # MRR
                # -------------------------

                rank_found = None

                for rank, chunk_id in enumerate(
                    retrieved_ids,
                    start=1
                ):
                    if chunk_id in expected_chunks:
                        rank_found = rank
                        break

                if rank_found is not None:
                    reciprocal_ranks.append(
                        1 / rank_found
                    )
                else:
                    reciprocal_ranks.append(0)

            total = len(evaluation_questions)

            recall = hits / total

            mrr = np.mean(reciprocal_ranks)

            metrics.append({
                "Method": method,
                "K": k,
                "Recall@K": recall,
                "MRR": mrr
            })

    return pd.DataFrame(metrics)

In [17]:
metrics_df = calculate_metrics(
    evaluation_questions,
    chunks_dict,
    chunk_embeddings,
    ground_truth
)

metrics_df

,Method,K,Recall@K,MRR
0,recursive,1,0.8,0.80
1,recursive,3,1.0,0.90
2,recursive,5,1.0,0.90
3,character,1,0.9,0.90
4,character,3,1.0,0.95
5,character,5,1.0,0.95
6,token,1,1.0,1.00
7,token,3,1.0,1.00
8,token,5,1.0,1.00
